# Инспектор конвертации Qwen -> D16Qwen и визуализация орбит $\mathbb{Z}_8 \rtimes \mathbb{Z}_2$

Этот блокнот визуализирует процесс сборки нативной модели `D16QwenForCausalLM` (`src/d16_qwen_model.py`) из весов вещественной сети Qwen с помощью конвертера `QwenToD16Converter` (`src/d16_qwen_converter.py`) и утилит `src/d16_vis_utils.py`:

In [ ]:
import importlib
import math
import os
from pathlib import Path
import sys
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Гарантируем наличие D:\repo\d16nn\qwen\src в sys.path после любого перезапуска ядра
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "ipynb" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import d16_complex
import d16_qwen_model
import d16_qwen_converter
import d16_vis_utils

importlib.reload(d16_complex)
importlib.reload(d16_qwen_model)
importlib.reload(d16_qwen_converter)
importlib.reload(d16_vis_utils)

from d16_complex import D16Complex, D16ComplexTensor, STATE_LABELS
from d16_qwen_model import D16QwenConfig, D16QwenForCausalLM
from d16_qwen_converter import OP_NAMES, QwenToD16Converter
from d16_vis_utils import (
    plot_layer_z8z2_spectrum,
    plot_learned_sector_gains,
    plot_single_orbit_compass,
    print_orbit_table,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Находим локальный чекпоинт Qwen2.5-0.5B
hf_home = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
hub_dir = hf_home / "hub" / "models--Qwen--Qwen2.5-0.5B"
snaps = list((hub_dir / "snapshots").glob("*")) if (hub_dir / "snapshots").is_dir() else []
snap_dir = next((s for s in snaps if (s / "config.json").is_file()), None)
assert snap_dir is not None, "Не найден локальный чекпоинт Qwen2.5-0.5B!"

qwen_tokenizer = AutoTokenizer.from_pretrained(str(snap_dir), local_files_only=True)

EVAL_TEXTS = [
    "The theory of general relativity describes gravitation as a geometric property of space and time.",
    "Complex numbers extend the real numbers by introducing the imaginary unit i, satisfying i^2 = -1.",
    "In deep neural networks, attention mechanisms route information across token positions.",
    "Water freezes at zero degrees Celsius and boils at one hundred degrees Celsius under standard pressure.",
]

GEN_PROMPTS = [
    "The capital of France is",
    "In mathematics, a complex number is",
]


@torch.no_grad()
def evaluate_d16_perplexity(model: D16QwenForCausalLM, tokenizer, texts, run_device) -> tuple[float, float]:
    model.eval()
    total_nll, total_tokens = 0.0, 0
    for text in texts:
        enc = tokenizer(text, return_tensors="pt")
        input_ids = enc["input_ids"].to(run_device)
        if input_ids.shape[1] < 2:
            continue
        out = model(input_ids=input_ids, labels=input_ids)
        num_t = input_ids.shape[1] - 1
        total_nll += float(out.loss.item()) * num_t
        total_tokens += num_t
    mean_nll = total_nll / max(total_tokens, 1)
    return mean_nll, math.exp(min(mean_nll, 20.0))


@torch.no_grad()
def run_d16_generation_test(model: D16QwenForCausalLM, tokenizer, prompts, run_device, max_new_tokens: int = 20):
    model.eval()
    for prompt in prompts:
        inp = tokenizer(prompt, return_tensors="pt")["input_ids"].to(run_device)
        out_ids = model.generate(
            inp, max_new_tokens=max_new_tokens, do_sample=False, eos_token_id=tokenizer.eos_token_id, use_cache=True
        )
        print(f"  Промпт : {prompt}")
        print(f"  Ответ  : {tokenizer.decode(out_ids[0].tolist(), skip_special_tokens=True)}")
        print("-" * 92)


# 3. Загружаем D16Qwen, если еще не загружен в память
if "d16_model" not in globals() or "conv_report" not in globals():
    print("Загружаем и конвертируем Qwen2.5-0.5B в нативный D16Qwen...")
    d16_model, conv_report = QwenToD16Converter.from_pretrained_dir(
        snap_dir, mode="native_d16", analyze_orbits=True, device="cpu"
    )
    d16_model = d16_model.to(device)

# 4. Проверка порога покоя 16: Indifferent (0xFFFF) в чистом режиме (0 склеенных пар)
QwenToD16Converter.activate_bitwise_orbit_links(d16_model, conv_report, pairs_per_layer=0)
print("=" * 92)
print("ПРОВЕРКА ПОРОГА ПОКОЯ 16:Indifferent (0xFFFF) НА СКОНВЕРТИРОВАННОМ Qwen2.5-0.5B")
print("=" * 92)
for eps in (0.0, 0.02, 0.05, 0.10, 0.15):
    d16_model.set_indifferent_threshold(eps)
    d16_model.set_collect_d16_stats(True, reset=True)
    nll_e, ppl_e = evaluate_d16_perplexity(d16_model, qwen_tokenizer, EVAL_TEXTS, device)
    sp_pct = d16_model.get_d16_state_summary()["indifferent_pct"]
    print(f"  eps = {eps:.2f} | Доля 0xFFFF: {sp_pct:5.2f}% | Mean NLL: {nll_e:.4f} | Perplexity: {ppl_e:.2f}")

d16_model.set_indifferent_threshold(0.0)
d16_model.set_collect_d16_stats(False)

# 5. Точное обращение матриц M_op^T (4<->5, 8<->11, 9<->10) и выбор партнера по косинусу
INVERSE_OP = torch.tensor([0, 1, 2, 3, 5, 4, 6, 7, 11, 10, 9, 8, 12, 13, 14, 15], dtype=torch.long)

@torch.no_grad()
def activate_true_cosine_orbit_pairs(
    model: D16QwenForCausalLM,
    report,
    pairs_per_layer: int = 32,
    min_orbit_score: float = 0.42,
) -> int:
    total_linked = 0
    for l_idx, layer in enumerate(model.model.layers):
        if pairs_per_layer <= 0 or l_idx not in report.layer_reports:
            layer.mlp.clear_orbit_links()
            continue

        rep = report.layer_reports[l_idx]
        # Выбираем из (m=1, 2, 3) партнера с максимальным НОРМИРОВАННЫМ косинусом в плоскости якоря
        member_sc = rep.member_scales[:, 1:].clamp_min(1e-8)            # [K, 3]
        cos_in_plane = (rep.local_2x2[:, 1:].norm(dim=(2, 3)) / (member_sc * (2.0 ** 0.5)))  # [K, 3]
        best_cos, best_idx = cos_in_plane.max(dim=1)                    # [K]

        order = torch.argsort(best_cos, descending=True)
        valid_order = order[best_cos[order] >= min_orbit_score][:pairs_per_layer]
        k = int(valid_order.numel())
        if k <= 0:
            layer.mlp.clear_orbit_links()
            continue

        m_choice = best_idx[valid_order] + 1  # в {1, 2, 3}
        anchors = rep.quadruplets[valid_order, 0]
        partners = rep.quadruplets[valid_order, m_choice]
        conv_ops = rep.group_ops[valid_order, m_choice]
        partner_ops = INVERSE_OP[conv_ops]
        scales = rep.member_scales[valid_order, m_choice]

        layer.mlp.set_orbit_links(
            partner_idx=partners,
            anchor_idx=anchors,
            op_ids=partner_ops,
            scales=scales,
        )
        total_linked += k
    return total_linked


print("\n" + "=" * 92)
print("ПРОВЕРКА ПОБИТОВЫХ ОРБИТ С ТОЧНЫМ ОБРАЩЕНИЕМ M_op^T И НОРМИРОВАННЫМ КОСИНУСОМ")
print("=" * 92)
for pairs_cnt, min_sc in [(16, 0.50), (32, 0.45), (64, 0.40), (128, 0.35)]:
    total_linked = activate_true_cosine_orbit_pairs(
        d16_model, conv_report, pairs_per_layer=pairs_cnt, min_orbit_score=min_sc
    )
    nll_k, ppl_k = evaluate_d16_perplexity(d16_model, qwen_tokenizer, EVAL_TEXTS, device)
    print(
        f"  Макс. {pairs_cnt:3d} пар/слой (min_cos={min_sc:.2f}) | "
        f"Связано пар по сети: {total_linked:4d} | Mean NLL: {nll_k:.4f} | PPL: {ppl_k:.2f}"
    )
    run_d16_generation_test(d16_model, qwen_tokenizer, GEN_PROMPTS, device, max_new_tokens=20)

QwenToD16Converter.activate_bitwise_orbit_links(d16_model, conv_report, pairs_per_layer=0)

In [ ]:
import importlib
import math
import os
from pathlib import Path
import torch
from transformers import AutoTokenizer

import d16_complex
import d16_qwen_model
import d16_qwen_converter
import d16_vis_utils

importlib.reload(d16_complex)
importlib.reload(d16_qwen_model)
importlib.reload(d16_qwen_converter)
importlib.reload(d16_vis_utils)

from d16_complex import D16Complex, D16ComplexTensor, STATE_LABELS
from d16_qwen_model import D16QwenConfig, D16QwenForCausalLM
from d16_qwen_converter import OP_NAMES, QwenToD16Converter
from d16_vis_utils import (
    plot_layer_z8z2_spectrum,
    plot_learned_sector_gains,
    plot_single_orbit_compass,
    print_orbit_table,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if "snap_dir" not in globals() or snap_dir is None:
    hf_home = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
    hub_dir = hf_home / "hub" / "models--Qwen--Qwen2.5-0.5B"
    snaps = list((hub_dir / "snapshots").glob("*")) if (hub_dir / "snapshots").is_dir() else []
    snap_dir = next((s for s in snaps if (s / "config.json").is_file()), None)

assert snap_dir is not None, "Не найден локальный чекпоинт Qwen2.5-0.5B!"
qwen_tokenizer = AutoTokenizer.from_pretrained(str(snap_dir), local_files_only=True)

EVAL_TEXTS = [
    "The theory of general relativity describes gravitation as a geometric property of space and time.",
    "Complex numbers extend the real numbers by introducing the imaginary unit i, satisfying i^2 = -1.",
    "In deep neural networks, attention mechanisms route information across token positions.",
    "Water freezes at zero degrees Celsius and boils at one hundred degrees Celsius under standard pressure.",
]

GEN_PROMPTS = [
    "The capital of France is",
    "In mathematics, a complex number is",
    "Water boils at a temperature of",
    "Artificial intelligence can help scientists by",
]


@torch.no_grad()
def evaluate_d16_perplexity(model: D16QwenForCausalLM, tokenizer, texts, run_device) -> tuple[float, float]:
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    for text in texts:
        enc = tokenizer(text, return_tensors="pt")
        input_ids = enc["input_ids"].to(run_device)
        if input_ids.shape[1] < 2:
            continue
        out = model(input_ids=input_ids, labels=input_ids)
        num_t = input_ids.shape[1] - 1
        total_nll += float(out.loss.item()) * num_t
        total_tokens += num_t
    mean_nll = total_nll / max(total_tokens, 1)
    return mean_nll, math.exp(min(mean_nll, 20.0))


@torch.no_grad()
def run_d16_generation_test(model: D16QwenForCausalLM, tokenizer, prompts, run_device, max_new_tokens: int = 20):
    model.eval()
    for prompt in prompts:
        inp = tokenizer(prompt, return_tensors="pt")["input_ids"].to(run_device)
        out_ids = model.generate(
            inp,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
        print(f"  Промпт : {prompt}")
        print(f"  Ответ  : {tokenizer.decode(out_ids[0].tolist(), skip_special_tokens=True)}")
        print("-" * 92)


if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Загружаем и конвертируем Qwen2.5-0.5B в нативный D16Qwen (mode='native_d16')...")
d16_model, conv_report = QwenToD16Converter.from_pretrained_dir(
    snap_dir, mode="native_d16", analyze_orbits=True, device="cpu"
)
d16_model = d16_model.to(device)

# 1. Базовый прогон: все 4864 объекта D16Complex в каждом слое независимы (pairs_per_layer=0)
QwenToD16Converter.activate_bitwise_orbit_links(d16_model, conv_report, pairs_per_layer=0)
nll_0, ppl_0 = evaluate_d16_perplexity(d16_model, qwen_tokenizer, EVAL_TEXTS, device)

print("=" * 92)
print(
    f"НАТИВНЫЙ D16Qwen (4864 -> {conv_report.d16_intermediate_size} объектов D16Complex, "
    f"0 склеенных пар) | Mean NLL: {nll_0:.4f} | Perplexity: {ppl_0:.2f}"
)
print("=" * 92)
run_d16_generation_test(d16_model, qwen_tokenizer, GEN_PROMPTS, device, max_new_tokens=20)

## Включение побитовых связей на орбитах $\mathbb{Z}_8 \rtimes \mathbb{Z}_2$ (activate_bitwise_orbit_links) и проверка генерации

In [ ]:
# Проверяем, как ведет себя сеть, когда партнерские объекты D16Complex на самых чистых орбитах
# берут (x, y) от якоря и получают свое состояние чистым побитовым сдвигом маски rotation!
for pairs_cnt, min_sc in [(32, 0.42), (64, 0.38), (128, 0.35), (256, 0.30)]:
    total_linked = QwenToD16Converter.activate_bitwise_orbit_links(
        d16_model,
        conv_report,
        pairs_per_layer=pairs_cnt,
        min_orbit_score=min_sc,
    )
    nll_k, ppl_k = evaluate_d16_perplexity(d16_model, qwen_tokenizer, EVAL_TEXTS, device)
    print("=" * 92)
    print(
        f"ПОБИТОВЫЕ ОРБИТЫ Z_8 |>< Z_2: макс. {pairs_cnt} пар/слой (min_score={min_sc:.2f}) | "
        f"Всего связано пар по сети: {total_linked} | Mean NLL: {nll_k:.4f} | PPL: {ppl_k:.2f}"
    )
    print("=" * 92)
    run_d16_generation_test(d16_model, qwen_tokenizer, GEN_PROMPTS[:2], device, max_new_tokens=20)

# Возвращаем все объекты в независимый режим после эксперимента
QwenToD16Converter.activate_bitwise_orbit_links(d16_model, conv_report, pairs_per_layer=0)

## Визуализация орбит, полярного 16-секторного компаса и отклика _eval_d16(z) при сдвиге rol(rotation, 2)

In [ ]:
import matplotlib.pyplot as plt

INSPECT_LAYER = min(8, conv_report.num_layers - 1)
layer_rep = conv_report.layer_reports[INSPECT_LAYER]

# 1. Таблица топ-10 орбит слоя (все 4864 объекта D16Complex сохранены)
print_orbit_table(layer_rep, top_k=10)

# 2. Круговой 16-секторный компас для двух лучших орбит слоя
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), subplot_kw={"projection": "polar"})
plot_single_orbit_compass(layer_rep, unit_idx=0, ax=axes[0])
plot_single_orbit_compass(layer_rep, unit_idx=1, ax=axes[1])
plt.tight_layout()
plt.show()

# 3. Спектр 16 операций Z_8 |>< Z_2 и послойная доля 45°-поворотов Адамара Re(z^2)
fig_spec = plot_layer_z8z2_spectrum(conv_report, layer_idx=INSPECT_LAYER)
plt.show()

# 4. Полярный отклик _eval_d16(z) для исходного rotation и при сдвиге маски rol(rotation, 2) (+45°)
d16_mlp_layer = d16_model.model.layers[INSPECT_LAYER].mlp
fig_resp = plot_learned_sector_gains(d16_mlp_layer, unit_indices=(0, 1, 2))
plt.show()

In [ ]:
# Точное соответствие между номером матрицы conv_op (0..15) в конвертере
# (где A_partner ≈ M_{conv_op}^T @ A_anchor) и индексом битовой операции в apply_all_16_d16_ops_u16:
CONV_OP_TO_PARTNER_BIT_OP = torch.tensor(
    [0, 4, 8, 12, 6, 2, 10, 14, 7, 5, 3, 1, 9, 11, 13, 15], dtype=torch.long
)

@torch.no_grad()
def activate_exact_bitwise_pairs(
    model: D16QwenForCausalLM,
    report,
    pairs_per_layer: int = 32,
    min_orbit_score: float = 0.42,
) -> int:
    total_linked = 0
    for l_idx, layer in enumerate(model.model.layers):
        if pairs_per_layer <= 0 or l_idx not in report.layer_reports:
            layer.mlp.clear_orbit_links()
            continue

        rep = report.layer_reports[l_idx]
        valid = rep.orbit_scores >= min_orbit_score
        k = min(pairs_per_layer, int(valid.sum().item()))
        if k <= 0:
            layer.mlp.clear_orbit_links()
            continue

        # Берем якорь (m=0) и его реального ближайшего партнера из тройки (m=1, 2, 3) по длине проекции в local_2x2
        loc_norms = rep.local_2x2[:k, 1:, :, :].norm(dim=(2, 3))  # [k, 3]
        best_m_offset = torch.argmax(loc_norms, dim=1) + 1         # в {1, 2, 3}
        row_arange = torch.arange(k)

        anchors = rep.quadruplets[row_arange, 0]
        partners = rep.quadruplets[row_arange, best_m_offset]
        conv_ops = rep.group_ops[row_arange, best_m_offset]
        bit_ops = CONV_OP_TO_PARTNER_BIT_OP[conv_ops]
        scales = rep.member_scales[row_arange, best_m_offset]

        layer.mlp.set_orbit_links(
            partner_idx=partners,
            anchor_idx=anchors,
            op_ids=bit_ops,
            scales=scales,
        )
        total_linked += k
    return total_linked


for pairs_cnt, min_sc in [(32, 0.42), (64, 0.38), (128, 0.35)]:
    total_linked = activate_exact_bitwise_pairs(d16_model, conv_report, pairs_per_layer=pairs_cnt, min_orbit_score=min_sc)
    nll_k, ppl_k = evaluate_d16_perplexity(d16_model, qwen_tokenizer, EVAL_TEXTS, device)
    print("=" * 92)
    print(
        f"ТОЧНЫЙ БИТОВЫЙ ИНДЕКС Z_8 |>< Z_2: макс. {pairs_cnt} пар/слой (min_score={min_sc:.2f}) | "
        f"Связано пар: {total_linked} | Mean NLL: {nll_k:.4f} | PPL: {ppl_k:.2f}"
    )
    print("=" * 92)
    run_d16_generation_test(d16_model, qwen_tokenizer, GEN_PROMPTS[:2], device, max_new_tokens=20)

QwenToD16Converter.activate_bitwise_orbit_links(d16_model, conv_report, pairs_per_layer=0)

In [ ]:
from transformers import AutoModelForCausalLM

# 1. Эталонный замер оригинального HF Qwen2.5-0.5B на тех же самых EVAL_TEXTS
hf_qwen = AutoModelForCausalLM.from_pretrained(str(snap_dir), local_files_only=True, torch_dtype=torch.float32).to(device)
hf_qwen.eval()

with torch.no_grad():
    tot_nll, tot_tok = 0.0, 0
    for text in EVAL_TEXTS:
        ids = qwen_tokenizer(text, return_tensors="pt")["input_ids"].to(device)
        out_hf = hf_qwen(input_ids=ids, labels=ids)
        n_t = ids.shape[1] - 1
        tot_nll += float(out_hf.loss.item()) * n_t
        tot_tok += n_t
    hf_nll = tot_nll / tot_tok
    hf_ppl = math.exp(hf_nll)

del hf_qwen
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Оригинальный HF Qwen2.5-0.5B на EVAL_TEXTS | Mean NLL: {hf_nll:.6f} | PPL: {hf_ppl:.4f}")

# 2. Наш нативный D16Qwen на тех же EVAL_TEXTS + проверка фильтрации шума в 0xFFFF (Indifferent)
QwenToD16Converter.activate_bitwise_orbit_links(d16_model, conv_report, pairs_per_layer=0)
print("-" * 86)
for eps in (0.0, 0.02, 0.05, 0.10, 0.15, 0.20):
    d16_model.set_indifferent_threshold(eps)
    d16_model.set_collect_d16_stats(True, reset=True)
    nll_d16, ppl_d16 = evaluate_d16_perplexity(d16_model, qwen_tokenizer, EVAL_TEXTS, device)
    sp_pct = d16_model.get_d16_state_summary()["indifferent_pct"]
    diff_ppl = ppl_d16 - hf_ppl
    print(f"D16Qwen (eps={eps:.2f}) | Доля 0xFFFF: {sp_pct:5.2f}% | Mean NLL: {nll_d16:.6f} | PPL: {ppl_d16:.4f} (Δ={diff_ppl:+.4f})")

d16_model.set_indifferent_threshold(0.0)
d16_model.set_collect_d16_stats(False)